In [ ]:
# !pip install torch
# !pip install --upgrade transformers
# !rm -rf /usr/local/lib/python3.11/dist-packages/~ransformers
# !pip uninstall transformers -y
# !pip uninstall huggingface_hub -y
# !pip cache purge
# !pip install transformers

In [ ]:
import torch
import torch.nn as nn
from transformers import BertModel, BertConfig
from torch.utils.data import Dataset, DataLoader
import os
import re
from pathlib import Path
from collections import defaultdict


In [24]:
def simplify_verilog_remove_port_names(input_path, output_path):
    with open(input_path, "r") as file:
        original_lines = file.readlines()

    # Preserve module line separately
    module_line = ""
    new_original_lines = []
    for line in original_lines:
        if line.strip().startswith("module "):
            module_line = line
        else:
            new_original_lines.append(line)

    # Remove the first 7 lines after module
    lines = new_original_lines[6:]

    buffer = ''
    inside_decl = False
    declared_lines = []
    new_lines = []

    # Group declaration and logic lines
    for line in lines:
        if re.match(r'^\s*(wire|input|output)', line) and ';' not in line:
            buffer = line
            inside_decl = True
        elif inside_decl:
            buffer += line
            if ';' in line:
                declared_lines.append(buffer)
                buffer = ''
                inside_decl = False
        elif not inside_decl and re.match(r'^\s*(wire|input|output).*;', line):
            declared_lines.append(line)
        else:
            new_lines.append(line)

    # Assign new names to all declared signals
    signal_map = {}
    cnt = 0
    renamed_lines = []

    for decl in declared_lines:
        match = re.match(r'^\s*(wire|input|output)(\s+\[.*?\])?\s+(.*);', decl.replace('\n', ' '))
        if not match:
            renamed_lines.append(decl)
            continue
        sig_type, bus, rest = match.groups()
        signals = [s.strip() for s in rest.split(',')]
        new_decls = []
        for sig in signals:
            original_sig = sig
            new_name = f"w_{cnt}"
            signal_map[original_sig] = new_name
            cnt += 1
            new_decls.append(new_name)
        renamed_lines.append((sig_type, bus if bus else '', new_decls))

    # Collapse declarations with same bus/type
    decl_by_type_bus = defaultdict(list)
    for sig_type, bus, names in renamed_lines:
        decl_by_type_bus[(sig_type, bus)].extend(names)

    collapsed_decls = []
    for (sig_type, bus), names in decl_by_type_bus.items():
        line = f"{sig_type} {bus} " if bus else f"{sig_type} "
        line += ', '.join(names) + ";"
        collapsed_decls.append(line)

    # Mapping gate types
    gate_type_map = {
        'DFFASRHQNx1_ASAP7_75t_R': 'dff',
        'AND2x2_ASAP7_75t_R': 'and',
        'OR2x2_ASAP7_75t_R': 'or',
        'INVx2_ASAP7_75t_R': 'not',
        'XOR2x1_ASAP7_75t_R': 'xor',
        'XNOR2x1_ASAP7_75t_R': 'xnor',
        'NOR2x1_ASAP7_75t_R': 'nor',
        'NAND2x1_ASAP7_75t_R': 'nand',
        'BUFx2_ASAP7_75t_R': 'buf'
    }
    dff_port_order = ['RN', 'SN', 'CK', 'D', 'Q']
    original_to_new = {
        'RESETN': 'RN',
        'SETN': 'SN',
        'CLK': 'CK',
        'D': 'D',
        'QN': 'Q'
    }

    gate_blocks = []
    buffer = ""
    in_gate = False

    # Extract gate blocks
    for line in new_lines:
        if re.match(r'^\s*\w+\s+\S+\s*\(.*', line):
            buffer = line
            in_gate = True
            if ');' in line:
                gate_blocks.append(buffer)
                buffer = ''
                in_gate = False
        elif in_gate:
            buffer += line
            if ');' in line:
                gate_blocks.append(buffer)
                buffer = ''
                in_gate = False
        else:
            gate_blocks.append(line)

    final_lines = []
    gate_cnt = 1
    default_const = "1'b1"

    for block in gate_blocks:
        if block.strip().startswith("module"):
            continue

        gate_match = re.match(r'^\s*(\w+)\s+(\S+)\s*\((.*)\);\s*$', block.replace('\n', ' ').strip())
        if gate_match:
            gate_type, gate_name, port_list = gate_match.groups()
            gate_type_simple = gate_type_map.get(gate_type, gate_type)
            gate_name = f"g_{gate_cnt}"
            gate_cnt += 1

            ports = port_list.split(',')
            port_dict = {}
            for port in ports:
                port = port.strip()
                p_match = re.match(r'\.(\w+)\((.*?)\)', port)
                if not p_match:
                    continue
                pname, sig = p_match.groups()
                sig = sig.strip()

                if sig.startswith('\\'):
                    lookup_key = sig
                else:
                    lookup_key = sig.split('[')[0] if '[' in sig else sig

                new_sig = signal_map.get(lookup_key, sig)
                if '[' in sig and not sig.startswith('\\'):
                    index = sig[sig.index('['):]
                    new_sig += index

                if gate_type_simple == 'dff' and pname in original_to_new:
                    port_dict[original_to_new[pname]] = new_sig
                else:
                    port_dict[pname] = new_sig

            # Format ports based on gate type
            if gate_type_simple == 'dff':
                new_ports = [f".{p}({port_dict.get(p, default_const)})" for p in dff_port_order]
            elif gate_type_simple in ['and', 'or', 'nor', 'nand', 'xor', 'xnor']:
                logic_order = ['Y', 'A', 'B']
                new_ports = [f"{port_dict.get(p, default_const)}" for p in logic_order]
            elif gate_type_simple in ['not', 'buf']:
                logic_order = ['Y', 'A']
                new_ports = [f"{port_dict.get(p, default_const)}" for p in logic_order]
            else:
                new_ports = [f".{k}({v})" for k, v in port_dict.items()]

            final_lines.append(f"{gate_type_simple} {gate_name} ( " + ', '.join(new_ports) + " );\n")
        else:
            final_lines.append(block)

    # Output final lines
    final_lines = [module_line] + collapsed_decls + final_lines

    with open(output_path, "w") as f:
        f.writelines(final_lines)

    print(f"Simplified file written to: {output_path}")

In [25]:
# folder_path = "./data/Training_Data/T8"
# destination_folder = "./data/Training_Data_Simplified/T8"

# for filename in os.listdir(folder_path):
#     file_path = os.path.join(folder_path, filename)
#     if os.path.isfile(file_path):  # ensures it's a file, not a subdirectory
#         print("Processing:", file_path)
#         output_path = os.path.join(destination_folder, filename)
#         simplify_verilog_remove_port_names(file_path, output_path)


# folder_path = "./data/Training_Data/T9"
# destination_folder = "./data/Training_Data_Simplified/T9"

# for filename in os.listdir(folder_path):
#     file_path = os.path.join(folder_path, filename)
#     if os.path.isfile(file_path):  # ensures it's a file, not a subdirectory
#         print("Processing:", file_path)
#         output_path = os.path.join(destination_folder, filename)
#         simplify_verilog_remove_port_names(file_path, output_path)


In [26]:
VOCAB = ['DFF', 'AND', 'OR', 'NOT', 'X', '[PAD]']
PAD_TOKEN = '[PAD]'
MAX_LEN = 512
TREE_DIM = 64
EMBED_DIM = 512
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [27]:
# Tokenizer
# ------------------------------
class SimpleGateTokenizer:
    def __init__(self, vocab):
        self.token2id = {tok: i for i, tok in enumerate(vocab)}
        self.pad_id = self.token2id[PAD_TOKEN]

    def encode(self, tokens, max_len=MAX_LEN):
        ids = [self.token2id.get(tok, self.token2id['X']) for tok in tokens]
        return ids[:max_len] + [self.pad_id] * (max_len - len(ids))

# ------------------------------
# Tree-Based Positional Encoding
# ------------------------------
def shift_right(vec, n=2):
    return [0]*n + vec[:-n]

def encode_node(parent_encoding, is_left):
    bit = [1, 0] if is_left else [0, 1]
    return (shift_right(parent_encoding)[:len(parent_encoding)] + bit)[:len(parent_encoding)]

def tree_based_positional_encoding(tree, tree_dim=TREE_DIM):
    encodings = []

    def traverse(node, encoding):
        encodings.append(encoding[:tree_dim])
        if 'left' in node:
            left_encoding = encode_node(encoding, is_left=True)
            traverse(node['left'], left_encoding)
        if 'right' in node:
            right_encoding = encode_node(encoding, is_left=False)
            traverse(node['right'], right_encoding)

    root_encoding = [0] * tree_dim
    traverse(tree, root_encoding)
    return encodings

# # ------------------------------
# # Dummy Tree and Tokens (Preorder)
# # ------------------------------
# sample_tree = {
#     'type': 'OR',
#     'left': {
#         'type': 'NOT',
#         'left': {'type': 'X'}
#     },
#     'right': {
#         'type': 'AND',
#         'left': {'type': 'X'},
#         'right': {'type': 'X'}
#     }
# }
# sample_tokens = ['OR', 'NOT', 'X', 'AND', 'X', 'X']

# ------------------------------
# Dataset
# ------------------------------
class GateDataset(Dataset):
    def __init__(self, samples, tokenizer):
        self.samples = samples
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        token_ids = self.tokenizer.encode(item['tokens'])
        tree_pos = item['tree_pos']
        tree_pos += [[0]*TREE_DIM] * (MAX_LEN - len(tree_pos))
        tree_pos = tree_pos[:MAX_LEN]
        return {
            'input_ids': torch.tensor(token_ids),
            'tree_pos': torch.tensor(tree_pos, dtype=torch.float32),
            'label': torch.tensor(item['label'])
        }

# ------------------------------
# Embedding Layer
# ------------------------------
class CustomEmbedding(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.word_embed = nn.Embedding(vocab_size, EMBED_DIM)
        self.pos_embed = nn.Embedding(MAX_LEN, EMBED_DIM)
        self.tree_proj = nn.Linear(TREE_DIM, EMBED_DIM)

    def forward(self, input_ids, tree_pos):
        word_emb = self.word_embed(input_ids)
        pos_ids = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        pos_emb = self.pos_embed(pos_ids)
        tree_emb = self.tree_proj(tree_pos)
        return word_emb + pos_emb + tree_emb

# ------------------------------
# BERT-based Trojan Classifier
# ------------------------------
class TrojanBERT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embeddings = CustomEmbedding(vocab_size)
        config = BertConfig(
            hidden_size=EMBED_DIM,
            num_hidden_layers=4,
            num_attention_heads=8,
            intermediate_size=1024,
            max_position_embeddings=MAX_LEN
        )
        self.bert = BertModel(config)
        self.classifier = nn.Linear(EMBED_DIM, 2)

    def forward(self, input_ids, tree_pos):
        x = self.embeddings(input_ids, tree_pos)
        out = self.bert(inputs_embeds=x)
        cls = out.last_hidden_state[:, 0]  # [CLS] token representation
        return self.classifier(cls)


In [28]:
# Build Fan-in Tree
def build_fanin_tree(parser, gate_name, max_depth=6):
    def recurse(g, depth):
        if depth == 0 or g not in parser.gates:
            return {'type': 'X'}

        gate = parser.gates[g]
        gate_type = gate.gate_type            # ✅ access attribute, not key
        inputs = gate.inputs             # ✅ access attribute, not key

        if len(inputs) == 0:
            return {'type': gate_type}

        if len(inputs) == 1:
            return {
                'type': gate_type,
                'left': recurse(inputs[0], depth - 1)
            }

        def to_binary_tree(ins):
            if len(ins) == 2:
                return {
                    'left': recurse(ins[0], depth - 1),
                    'right': recurse(ins[1], depth - 1)
                }
            return {
                'left': recurse(ins[0], depth - 1),
                'right': to_binary_tree(ins[1:])
            }

        tree = {'type': gate_type}
        tree.update(to_binary_tree(inputs))
        return tree

    return recurse(gate_name, max_depth)

In [29]:
tokenizer = SimpleGateTokenizer(VOCAB)


from exploit_gates import NetlistParser
from dataset_utils import GateEntry
import random


Trojan_number = 8
trojan_data = []

folder_path = "./data/Training_Data_Simplified/T8"

for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    if os.path.isfile(file_path):  # ensures it's a file, not a subdirectory
        print("Processing:", file_path)
        parser_trojan = NetlistParser()
        parser_trojan.parse_netlist(file_path)
        selected_gates_trojan = [
            gname for gname in parser_trojan.gates
            #if not has_primary_inputs_or_constants(parser, gname)
        ]
        for gate_name in selected_gates_trojan:
            tokens = parser_trojan.tokenize_gate_fanin_cone(gate_name, max_depth=6)
            tree = build_fanin_tree(parser_trojan, gate_name)
            tree_pos = tree_based_positional_encoding(tree)
            trojan_data.append({
                'tokens': tokens,
                'tree_pos': tree_pos,
                'label': 1
            })
        # if len(trojan_data) %10 == 0:
        #     print(f"Gate: {gate_name}, Tokens: {tokens}, Tree Pos: {tree_pos}")

folder_path = "./data/Training_Data_Simplified/T9"

for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    if os.path.isfile(file_path):  # ensures it's a file, not a subdirectory
        print("Processing:", file_path)
        parser_trojan = NetlistParser()
        parser_trojan.parse_netlist(file_path)
        selected_gates_trojan = [
            gname for gname in parser_trojan.gates
            #if not has_primary_inputs_or_constants(parser, gname)
        ]
        for gate_name in selected_gates_trojan:
            tokens = parser_trojan.tokenize_gate_fanin_cone(gate_name, max_depth=6)
            tree = build_fanin_tree(parser_trojan, gate_name)
            tree_pos = tree_based_positional_encoding(tree)
            trojan_data.append({
                'tokens': tokens,
                'tree_pos': tree_pos,
                'label': 1
            })
        # if len(trojan_data) %10 == 0:
        #     print(f"Gate: {gate_name}, Tokens: {tokens}, Tree Pos: {tree_pos}")

    


normal_data = []

for Trojan_less_number in range(10):
    parser_normal = NetlistParser()
    parser_normal.parse_netlist(f"./data/Training_Data_Normal/test_design{Trojan_less_number+20}.v")
    eligible_gates = [
        gname for gname in parser_normal.gates
        #if not has_primary_inputs_or_constants(parser_normal, gname)
    ]
    sample_size = len(eligible_gates) // 2
    selected_gates_normal = random.sample(eligible_gates, sample_size)

    for gate_name in selected_gates_normal:
        tokens = parser_normal.tokenize_gate_fanin_cone(gate_name, max_depth=6)
        tree = build_fanin_tree(parser_normal, gate_name)
        tree_pos = tree_based_positional_encoding(tree)
        normal_data.append({
            'tokens': tokens,
            'tree_pos': tree_pos,
            'label': 0
        })


print(len(trojan_data))
print(len(normal_data))




Processing: ./data/Training_Data_Simplified/T8/trojan8_bit0_0_bit2_0.vg
Processing: ./data/Training_Data_Simplified/T8/trojan8_bit0_1_bit1_0_bit2_1.vg
Processing: ./data/Training_Data_Simplified/T8/trojan8_bit0_0_bit1_0_bit2_1.vg
Processing: ./data/Training_Data_Simplified/T8/trojan8_bit1_1.vg
Processing: ./data/Training_Data_Simplified/T8/Trojan8.vg
Processing: ./data/Training_Data_Simplified/T8/trojan8_bit0_0_bit1_0.vg
Processing: ./data/Training_Data_Simplified/T8/trojan8_bit0_0_bit1_1.vg
Processing: ./data/Training_Data_Simplified/T8/trojan8_bit0_0_bit1_1_bit2_1.vg
Processing: ./data/Training_Data_Simplified/T8/trojan8_bit0_0.vg
Processing: ./data/Training_Data_Simplified/T8/trojan8_bit0_1_bit2_0.vg
Processing: ./data/Training_Data_Simplified/T8/trojan8_bit0_0_bit2_1.vg
Processing: ./data/Training_Data_Simplified/T8/trojan8_bit0_0_bit1_1_bit2_0.vg
Processing: ./data/Training_Data_Simplified/T8/trojan8_bit2_1.vg
Processing: ./data/Training_Data_Simplified/T8/trojan8_bit0_1_bit1_1.vg

In [30]:
if len(trojan_data) > len(normal_data):
    normal_data = normal_data * (len(trojan_data) // len(normal_data) + 1)
full_dataset = trojan_data + normal_data
print(f"Number of sampled normal gates: {len(normal_data)}")
print(f"Total dataset size: {len(full_dataset)}")

Number of sampled normal gates: 112250
Total dataset size: 223991


In [31]:
from tqdm import tqdm


dataset = GateDataset(full_dataset, tokenizer)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

model = TrojanBERT(vocab_size=len(VOCAB)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.CrossEntropyLoss()

# # Train 1 step (demo only)
# model.train()
# for batch in dataloader:
#     input_ids = batch['input_ids'].to(device)
#     tree_pos = batch['tree_pos'].to(device)
#     labels = batch['label'].to(device)

#     logits = model(input_ids, tree_pos)
#     loss = loss_fn(logits, labels)

#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()

for epoch in range(4):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader):
        input_ids = batch['input_ids'].to(device)
        tree_pos = batch['tree_pos'].to(device)
        labels = batch['label'].to(device)

        logits = model(input_ids, tree_pos)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch + 1}, Loss: {total_loss:.4f}")


print("✅ Training complete.")

  0%|          | 0/27999 [00:00<?, ?it/s]

100%|██████████| 27999/27999 [16:19<00:00, 28.58it/s]


Epoch 1, Loss: 1409.4562


100%|██████████| 27999/27999 [16:20<00:00, 28.55it/s]


Epoch 2, Loss: 1146.2411


100%|██████████| 27999/27999 [16:19<00:00, 28.57it/s]


Epoch 3, Loss: 1113.6235


100%|██████████| 27999/27999 [16:19<00:00, 28.58it/s]

Epoch 4, Loss: 1105.2433
✅ Training complete.


In [32]:
import torch.nn.functional as F

def predict_gate_trojan_probability(parser, gate_name, model, tokenizer):
    # Step 1: Token sequence
    tokens = parser.tokenize_gate_fanin_cone(gate_name, max_depth=6)

    # Step 2: Fan-in tree
    tree = build_fanin_tree(parser, gate_name, max_depth=6)

    # Step 3: Tree-based position encoding
    tree_pos = tree_based_positional_encoding(tree)

    # Step 4: Padding to MAX_LEN
    token_ids = tokenizer.encode(tokens, max_len=MAX_LEN)
    tree_pos += [[0]*TREE_DIM] * (MAX_LEN - len(tree_pos))
    tree_pos = tree_pos[:MAX_LEN]

    # Step 5: Tensorize and move to device
    input_ids = torch.tensor([token_ids]).to(device)
    tree_pos_tensor = torch.tensor([tree_pos], dtype=torch.float32).to(device)

    # Step 6: Run through model
    model.eval()
    with torch.no_grad():
        logits = model(input_ids, tree_pos_tensor)
        probs = F.softmax(logits, dim=-1)
        trojan_prob = probs[0][1].item()  # class 1 = Trojan
        #print("Model output logits:", logits)
        #print("Probabilities:", probs)

    return trojan_prob

In [33]:
#save the model with DFF in vocab
model_save_path = f"./models1/TrojanBERT_Trojan_8_and_9.pt"
torch.save(model.state_dict(), model_save_path)

In [34]:
# Extract Trojan gates from the reference file
def extract_trojan_gates(filename):
    trojan_gates = []
    inside_block = False

    with open(filename, 'r') as file:
        for line in file:
            stripped = line.strip()
            if stripped == "TROJAN_GATES":
                inside_block = True
                continue
            if stripped == "END_TROJAN_GATES":
                break
            if inside_block:
                trojan_gates.append(stripped)

    return trojan_gates

In [35]:
import torch.nn.functional as F

def predict_gate_trojan_probability(parser, gate_name, model, tokenizer):
    # Step 1: Token sequence
    tokens = parser.tokenize_gate_fanin_cone(gate_name, max_depth=6)

    # Step 2: Fan-in tree
    tree = build_fanin_tree(parser, gate_name, max_depth=6)

    # Step 3: Tree-based position encoding
    tree_pos = tree_based_positional_encoding(tree)

    # Step 4: Padding to MAX_LEN
    token_ids = tokenizer.encode(tokens, max_len=MAX_LEN)
    tree_pos += [[0]*TREE_DIM] * (MAX_LEN - len(tree_pos))
    tree_pos = tree_pos[:MAX_LEN]

    # Step 5: Tensorize and move to device
    input_ids = torch.tensor([token_ids]).to(device)
    tree_pos_tensor = torch.tensor([tree_pos], dtype=torch.float32).to(device)

    # Step 6: Run through model
    model.eval()
    with torch.no_grad():
        logits = model(input_ids, tree_pos_tensor)
        probs = F.softmax(logits, dim=-1)
        trojan_prob = probs[0][1].item()  # class 1 = Trojan
        #print("Model output logits:", logits)
        #print("Probabilities:", probs)

    return trojan_prob

In [39]:
parser_eval = NetlistParser()
evaluation_design_number = 9
parser_eval.parse_netlist(f"./data/Evaluation_Data/evaluation_new/simplified_design{evaluation_design_number}.v")
#parser_eval.parse_netlist(f"./data/simplified_trojans/Trojan{Trojan_number}_simplified.v")

gate = 'g3851'
prob = predict_gate_trojan_probability(parser_eval, gate, model, tokenizer)
print(f"Trojan probability for {gate}: {prob:.4f}")

results = []
for gate_name in parser_eval.gates:
    prob = predict_gate_trojan_probability(parser_eval, gate_name, model, tokenizer)
    results.append((gate_name, prob))

results.sort(key=lambda x: x[1], reverse=True)
print("Top 10 gates with highest Trojan probability:")
for gate_name, prob in results[:10]:
    print(f"{gate_name}: {prob:.4f}")


Trojan probability for g3851: 0.9999
Top 10 gates with highest Trojan probability:
g1555_1: 0.7965
g1555_2: 0.4025
g146: 0.2350
g254: 0.2350
g1069: 0.2350
g1119: 0.2350
g3870: 0.2350
g4142: 0.2350
g1587_2: 0.0723
g2606_1: 0.0659


In [44]:
reference_Trojans_file = f"./data/Evaluation_Data/release_all2/trojan/result{evaluation_design_number}.txt"
actual_trojan_gates = extract_trojan_gates(reference_Trojans_file)
print(f"Number of trojan gates in reference file: {len(actual_trojan_gates)}")


predict_Trojan_threshold = 0.0001 # change this to the threshold you want to use
predicted_trojan_gates = set(gate_name.split('_')[0] for gate_name, prob in results if prob >= predict_Trojan_threshold)
negatives = set(gate_name.split('_')[0] for gate_name, prob in results if prob < predict_Trojan_threshold)
positives = set(gate_name.split('_')[0] for gate_name, prob in results if prob >= predict_Trojan_threshold)
actual_normal_gates = []
print (f"Number of predicted trojan gates: {len(predicted_trojan_gates)}")

parser_eval_actual = NetlistParser()
parser_eval_actual.parse_netlist(f"data/Evaluation_Data/release_all2/trojan/design{evaluation_design_number}.v")

for gate_name in parser_eval_actual.gates:
    if gate_name not in actual_trojan_gates:
        actual_normal_gates.append(gate_name)

true_negatives = set(actual_normal_gates).intersection(negatives)
true_positives = set(actual_trojan_gates).intersection(positives)
false_negatives = negatives - set(actual_normal_gates)
false_positives = positives - set(actual_trojan_gates)



print (f"Actual Trojan gates: {sorted(actual_trojan_gates, reverse=True)}")
print (f"Predicted Trojan gates: {sorted(predicted_trojan_gates, key=lambda x: (len(x), x), reverse=True)}")

true_negative = len(true_negatives)
true_positive = len(true_positives)
false_positive = len(false_positives)
false_negative = len(false_negatives)
TPR = true_positive / len(actual_trojan_gates) if actual_trojan_gates else 0
FPR = false_positive / (len(predicted_trojan_gates) + false_negative) if (len(predicted_trojan_gates) + false_negative) > 0 else 0
precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
f1_score = 2 * (recall * precision) / (recall + precision) if (recall + precision) > 0 else 0


print(f"True Positive Rate (TPR): {TPR:.4f}")
print(f"False Positive Rate (FPR): {FPR:.4f}")
print(f"True Positive Count: {true_positive}")
print(f"False Positive Count: {false_positive}")
print(f"False Negative Count: {false_negative}")
print(f"True Negative Count: {len(true_negatives)}")
print (f"precision: {precision:.4f}")
print (f"recall: {TPR:.4f}")
print (f"F1 score: {f1_score:.4f}")
print (f"total number of gates: {len(results)}")
print_excel_outputs = [true_positive, false_positive, false_negative, len(true_negatives), precision, TPR, f1_score, predict_Trojan_threshold]
print('\t'.join(map(str, print_excel_outputs)))

Number of trojan gates in reference file: 2002
Number of predicted trojan gates: 966
Actual Trojan gates: ['g999', 'g998', 'g997', 'g995', 'g992', 'g991', 'g990', 'g99', 'g988', 'g987', 'g985', 'g984', 'g981', 'g978', 'g976', 'g975', 'g973', 'g972', 'g968', 'g967', 'g966', 'g965', 'g962', 'g960', 'g958', 'g952', 'g946', 'g942', 'g941', 'g938', 'g935', 'g934', 'g933', 'g932', 'g931', 'g930', 'g929', 'g928', 'g926', 'g922', 'g920', 'g92', 'g914', 'g912', 'g911', 'g910', 'g91', 'g908', 'g905', 'g903', 'g901', 'g90', 'g9', 'g899', 'g898', 'g896', 'g895', 'g894', 'g893', 'g892', 'g891', 'g89', 'g888', 'g887', 'g883', 'g882', 'g878', 'g877', 'g876', 'g875', 'g874', 'g872', 'g871', 'g869', 'g865', 'g855', 'g851', 'g848', 'g847', 'g844', 'g842', 'g841', 'g84', 'g839', 'g838', 'g837', 'g835', 'g834', 'g833', 'g832', 'g831', 'g83', 'g826', 'g825', 'g824', 'g823', 'g821', 'g820', 'g818', 'g816', 'g815', 'g814', 'g810', 'g805', 'g801', 'g80', 'g8', 'g798', 'g792', 'g791', 'g790', 'g79', 'g788', 'g